# Deliverable 2 — EDA, Data Leakage & Choosing the Right Model

**ADSC × American Airlines — Fall 2026**

In Deliverable 1 you were handed one model — `XGBRegressor` — and you trained it. It worked, and you got an R² somewhere north of 0.97.

Two questions were never asked:

1. **Was that score real?** (You cannot currently answer that. Section 3.)
2. **Was XGBoost the right choice?** (You have no idea, because you never tried anything else.)

This deliverable answers both. You will go deeper on EDA, learn what **data leakage** is and measure what it cost you, then run a genuine model-selection exercise: **Lasso vs Ridge**, **Random Forest vs XGBoost**, and **ARIMA** — plus the choice that sits underneath all of them and that nobody teaches you to make deliberately, your **loss function**.

---

## What you will learn

| Section | Topic | The question it answers |
|---|---|---|
| **2** | **Deeper EDA** | What does the data actually look like, and what does that imply about which model to reach for? |
| **3** | **Data leakage** | Why was the Deliverable 1 score too good, and how do I stop it happening again? |
| **4** | **Lasso vs Ridge** | Two linear models, two penalties. When do I want shrinkage, and when do I want feature selection? |
| **5** | **Loss functions** | MSE, MAE, Huber, quantile — what am I actually asking the model to minimise, and does it match what the business cares about? |
| **6** | **XGBoost vs Random Forest vs ARIMA** | Three model families. Which fits *this* problem, and why is one of them answering a completely different question? |
| **7** | **Pick one and defend it** | Given everything above, what would you ship to American Airlines? |

The single most important idea in here: **model selection is not a leaderboard.** It is a set of choices — model family, loss function, regularisation — each of which should follow from something you observed in the data or something the business told you. By the end you should be able to justify every one of your choices without using the phrase "it scored the best".

---

## What we are building on from Deliverable 1

We assume all of this and will not re-explain it:

- loading the CSV, `.shape` / `.info()` / `.describe()`
- null counts, mean / median / std / quartiles / IQR with NumPy
- the 1.5 × IQR outlier rule
- the price histogram, price-by-airline box plot, duration-vs-price scatter
- `LabelEncoder`, an 80/20 `train_test_split`, `XGBRegressor`
- MAE, RMSE, R², and gain-based feature importance

Keep **your own completed Deliverable 1 notebook** open in another tab — it is in `deliverable1/`, saved under your name — because we refer back to it by section number throughout.

---

## The git workflow changed — read this before you touch anything

Deliverable 1 was solo: you cut `deliverable1/<your-name>` and opened a PR straight into `main`. With 70+ members that does not scale, and it is not how real teams work either.

**From Deliverable 2 onward you work in a team, and your team owns a branch tree.**

```
main                          <- protected. Only a PM merges here.
 └── team-06/main             <- YOUR TEAM's integration branch ("team trunk")
      ├── team-06/alex-chen    <- a member branch: your own work
      ├── team-06/priya-r      <- another member, in parallel
      └── team-06/sam-lee
```

**What matters is the structure, not the names.** Three rules:

1. **Your team has one trunk branch.** The suggested name is `team-##/main` — your team number, zero-padded, so team 6 is `team-06`. If that exact name is unavailable (see the note below), any name your team agrees on is completely fine.
2. **You personally work on your own branch off that trunk**, e.g. `team-##/<your-name>`. Never commit directly to the trunk.
3. **Work moves up the tree:** your branch → your team's trunk → `main`. Your PR goes into **the trunk**, reviewed by a teammate. Only the trunk opens a PR into `main`, and only a PM merges that one.

### Setting it up

**One person on the team** creates the trunk (do this once, then tell your teammates it exists):

```sh
git switch main
git pull origin main
git switch -c team-06/main          # use YOUR team number
git push -u origin team-06/main
```

**Every member** then branches off the team trunk — not off `main`:

```sh
git fetch origin
git switch -c team-06/alex-chen origin/team-06/main
git push -u origin team-06/alex-chen
```

### Submitting

```sh
git add deliverable2/flight_price_modeling.ipynb
git commit -m "deliverable2: EDA, leakage fixes, model selection"
git push

# then on GitHub: open a PR from team-06/alex-chen  ->  team-06/main
# (base branch is your TEAM's trunk, NOT the repo's main)
```

> **⚠️ If `git switch -c team-##/main` fails, this is why.** Git stores branch names as file paths, so a branch name cannot be a *prefix* of another. If a plain branch named `team-07` already exists, then `team-07/main` is **impossible** — git says `cannot lock ref ... 'team-07' exists`. Several teams are in exactly this position from earlier in the term.
>
> **This is not a problem, and nothing needs deleting.** Just pick a name that is free and tell your team what it is:
>
> ```sh
> git switch -c team-07-main      # a dash instead of a slash -- always works
> ```
>
> The same rule applies to member branches: `team-06/alex-chen` and `team-06/alex-chen-v2` are fine, but `team-06/alex-chen/fix` would be blocked by `team-06/alex-chen`. Keep each name a flat leaf.

> **⚠️ Notebooks and merge conflicts.** A `.ipynb` is JSON, and git conflicts on it are genuinely horrible. Two habits make it survivable:
> - **Clear outputs before every commit** (Kernel → Restart & Clear Output). Most notebook conflicts are rendered output, not code.
> - **Each member completes their own copy of this notebook.** You are *not* meant to split the cells across teammates. The notebook is individual learning; the shared team artifact is `deliverable2/TEAM_REPORT.md`, which your team fills in together on the team trunk.

Full detail, including each member's named review duty, is in [`deliverable2/README.md`](README.md). The general workflow rules have not changed — [`CONTRIBUTING.md`](../CONTRIBUTING.md) is still the operating manual.

---

> ### 🗒️ A note on AI tools
>
> Half of this deliverable is *judgment*: is this leakage, is Lasso the right penalty, does ARIMA even apply here. An AI will hand you a confident answer to every one of those, and you will have learned nothing and be unable to defend it when a teammate asks in review.
>
> **Use AI as a tutor, not an author:**
> - "Why does Lasso drive coefficients to exactly zero when Ridge does not?"
> - "What does `objective='reg:pseudohubererror'` change about training?"
> - "Why is ARIMA inappropriate for data without a time index?"
>
> **You must be able to explain every line you submit.** A teammate *will* review your PR and *will* ask.

---

## How to approach this deliverable

Read this before you start. It will save you more time than it costs.

### 1. The written answers are the deliverable — not the plots

Every section ends with a markdown prompt asking *why*. Those are the thing being assessed. A plot with no conclusion attached is worth nothing, and a number with no interpretation is worth less than no number at all, because it invites someone to act on it.

Concretely: when a cell tells you to make a histogram, the deliverable is not the histogram. It is the sentence **"...therefore I should \_\_\_ when I model."** If you cannot finish that sentence, you have not finished the cell.

### 2. The honest score is the goal, whichever direction it moves

Section 3 removes three leaks from your Deliverable 1 setup and measures what each was worth. **Do not assume the number must fall.** On some datasets a leak is worth a lot of fake R² and the honest score drops sharply; on others it turns out to have been adding almost nothing, and your score barely moves or even ticks up.

Both outcomes are real results. **The point is not that your score goes down — it is that you can no longer tell which it will be until you remove the leaks and measure.** An unmeasured score is untrustworthy no matter how good it looks.

So: report whatever you get, and never tune your way toward a number you were hoping for. If your honest R² does drop, being able to explain why the lower number is the real one is the single most professional thing a data scientist does. If it does not drop, say so and explain why that leak mattered less here than it might have.

### 3. How to think about choosing a model

The instinct from Deliverable 1 is "run them all, keep the best score". Resist it. Work through the questions in this order, because each one can eliminate options before you spend compute on them:

1. **What shape is my data, and what shape does this model require?** This is first because it is disqualifying. ARIMA needs an ordered time series; if you do not have one, no amount of tuning helps. Section 6 is built around this question.
2. **What am I asking the model to minimise?** The default is squared error, which is a choice you have been making silently. Section 5 is about making it on purpose.
3. **What does the stakeholder need?** A client who needs to *explain* a price to a customer may be better served by a Ridge model with readable coefficients than by a boosted ensemble that scores two points higher. Accuracy is one requirement among several.
4. **What does it cost?** Fit time, tuning burden, how many ways it can break in production. A model that takes 40× longer for 1% more R² is usually the wrong call.
5. **Only then: what scores best?**

If your final justification is "it had the highest R²", you have skipped the deliverable.

### 4. How to work the code

- **Run cells in order, top to bottom.** Later sections depend on variables defined earlier (`clean`, `X_train`, `prep_tree`). If something is `None` or undefined, you skipped a cell.
- **Check your work as you go.** After any transform, look at the result — `.head()`, `.shape`, `.value_counts()`. Most bugs in data code are silent: the wrong answer computes perfectly happily.
- **Some cells are slow.** Random Forest on 240k rows and 5-fold CV both take minutes. If a cell is painful on your machine, **subsample it and say so in your PR** — that is a normal engineering decision, not cheating. `X.sample(80_000, random_state=RANDOM_STATE)` is your friend.
- **Read the error, then the line above it.** Python tracebacks name the failing line. Nine times in ten the actual mistake is one line earlier.
- **Commit per section**, not in one lump at the end. Small commits are the entire reason your history is useful to a reviewer.

### 5. How to work with your team

Each member completes **their own full copy** of this notebook — the work is individual, because the learning is individual. What your team does **together** is agree on the conclusions and write them into `TEAM_REPORT.md` on your team's trunk branch.

Beyond that, how your team organises itself is up to your team. Decide in your first meeting and write the decision down somewhere your PM can see it.

One piece of hard-won advice: **clear your outputs before every commit** (Kernel → Restart & Clear Output). Notebook merge conflicts are mostly conflicts in rendered output, not in code, and this one habit prevents the large majority of them.

---

## References

Every link below was checked and goes to the specific page you need — not a search result or a landing page. Keep the ones for your current section open in browser tabs.

### Start here

| | |
|---|---|
| **Dataset** — Flight Price Prediction (EaseMyTrip) | https://www.kaggle.com/datasets/shubhambathwal/flight-price-prediction |
| **scikit-learn: Common pitfalls and recommended practices** — the single most useful page for this deliverable. Shows a "Wrong" and a "Right" version of each mistake. | https://scikit-learn.org/stable/common_pitfalls.html |
| **An Introduction to Statistical Learning** — the standard undergraduate text, **free PDF**. Ch. 6 covers Ridge/Lasso, Ch. 8 covers trees and boosting. Python edition (ISLP) also available. | https://www.statlearning.com/ |

### Section 3 — leakage & validation

| | |
|---|---|
| Data leakage, with a worked example of a score collapsing from 0.76 to chance once the leak is fixed | https://scikit-learn.org/stable/common_pitfalls.html#data-leakage |
| Inconsistent preprocessing — why the transform must be fitted once and applied, not refitted | https://scikit-learn.org/stable/common_pitfalls.html#inconsistent-preprocessing |
| Pipelines and composite estimators (`Pipeline`, `ColumnTransformer`) | https://scikit-learn.org/stable/modules/compose.html |
| Cross-validation: evaluating estimator performance | https://scikit-learn.org/stable/modules/cross_validation.html |

### Section 4 — Lasso vs Ridge

| | |
|---|---|
| Ridge regression — §1.1.2 of the Linear Models guide | https://scikit-learn.org/stable/modules/linear_model.html#ridge-regression-and-classification |
| Lasso — §1.1.3, including the note on why the objective is scaled by `1/(2·n_samples)` | https://scikit-learn.org/stable/modules/linear_model.html#lasso |
| Elastic-Net — §1.1.5 | https://scikit-learn.org/stable/modules/linear_model.html#elastic-net |
| **Common pitfalls in interpreting linear coefficients** — worked example of why scale matters, why coefficients are conditional not marginal, and how correlated features make coefficients unstable. Directly relevant to reading your Lasso output. | https://scikit-learn.org/stable/auto_examples/inspection/plot_linear_model_coefficient_interpretation.html |

### Section 5 — loss functions

| | |
|---|---|
| XGBoost learning-task parameters — the full `objective` list (`reg:squarederror`, `reg:absoluteerror`, `reg:pseudohubererror`, `reg:quantileerror`) | https://xgboost.readthedocs.io/en/stable/parameter.html#learning-task-parameters |
| **Prediction intervals for gradient boosting regression** — the reference implementation for Section 5's quantile models, including the coverage check. Note its own conclusion: *each model performs best on the metric it was trained for.* | https://scikit-learn.org/stable/auto_examples/ensemble/plot_gradient_boosting_quantile.html |
| Regression metrics — MAE, RMSE, R², and `mean_pinball_loss` | https://scikit-learn.org/stable/modules/model_evaluation.html#regression-metrics |

### Section 6 — model families

| | |
|---|---|
| **Introduction to Boosted Trees** — the clearest explanation of additive training anywhere, and it explicitly notes that random forests and boosted trees share the same model form and differ in *how they are trained*. Read this before writing your bagging-vs-boosting answer. | https://xgboost.readthedocs.io/en/stable/tutorials/model.html |
| Random forests and randomized tree ensembles — §1.11.2 | https://scikit-learn.org/stable/modules/ensemble.html#random-forests |
| Gradient-boosted trees — §1.11.1 | https://scikit-learn.org/stable/modules/ensemble.html#gradient-boosted-trees |
| `statsmodels` ARIMA API — the `order=(p, d, q)` signature | https://www.statsmodels.org/stable/generated/statsmodels.tsa.arima.model.ARIMA.html |
| **Forecasting: Principles and Practice, Ch. 9 — ARIMA models.** Free online, and the standard reference. | https://otexts.com/fpp3/arima.html |
| §9.1 Stationarity and differencing — read this before choosing your `d` | https://otexts.com/fpp3/arima.html#sec-stationarity |

---

## Setup

Copy the dataset you already downloaded to **`deliverable2/data/Clean_Dataset.csv`**. That path is git-ignored; **do not commit the CSV.**

One new package this time — `statsmodels`, for ARIMA:

```sh
pip install pandas numpy matplotlib seaborn scikit-learn xgboost statsmodels
```

---
# Section 0 — Imports & helpers

Run this cell first. Everything below depends on it.

Some cells in this notebook are **given to you complete** — the helpers, and the reproduction of your Deliverable 1 baseline. Read them; they are the reference implementations you will copy patterns from. The cells marked `# TODO` are yours.

In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- splitting & validation ------------------------------------------------
from sklearn.model_selection import train_test_split, KFold, cross_validate

# --- preprocessing ---------------------------------------------------------
# OneHotEncoder  : one 0/1 column per category. No invented ordering.
# OrdinalEncoder : integers in an order YOU specify (zero < one < two_or_more).
# StandardScaler : centres and scales numerics. Optional for trees,
#                  MANDATORY for Lasso/Ridge -- Section 4 explains why.
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

# --- composition -----------------------------------------------------------
# Pipeline guarantees every transform is fitted on the TRAINING fold only.
# That is our main defence against the leakage in Section 3.
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# --- the models we will compare -------------------------------------------
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

# --- metrics ---------------------------------------------------------------
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                             r2_score, mean_pinball_loss)

# --- display ---------------------------------------------------------------
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='whitegrid')

RANDOM_STATE = 42   # every split, every model. Reproducibility is not optional.
DATA_PATH = 'deliverable2/data/Clean_Dataset.csv'   # relative to the repo root
import os
if not os.path.exists(DATA_PATH):              # launched from inside deliverable2/
    DATA_PATH = 'data/Clean_Dataset.csv'      # (VS Code's default notebook cwd)

print('pandas', pd.__version__, '| numpy', np.__version__)
import sklearn; print('scikit-learn', sklearn.__version__)
import xgboost; print('xgboost', xgboost.__version__)

# statsmodels is only needed in Section 6c (ARIMA). If this import fails,
# `pip install statsmodels` and restart the kernel.
import statsmodels.api as sm
print('statsmodels', sm.__version__)

In [ ]:
def regression_report(y_true, y_pred, label=''):
    """Return a dict of the four metrics we use everywhere in this notebook.

    MAE   -- average absolute miss, in rupees. Interpretable.
    RMSE  -- squared-error based, so large misses dominate. RMSE >> MAE means a
             few very bad predictions are hiding behind a decent average.
    R2    -- fraction of price variance explained.
    MAPE% -- average miss as a PERCENTAGE of true price. The business metric: being
             off by 2,000 INR is trivial on a 52,000 ticket and awful on a 6,000 one.
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return {
        'model': label,
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'R2': r2_score(y_true, y_pred),
        'MAPE%': float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100),
    }


# Running scoreboard. Every model we fit goes on here, so at the end we can read
# the whole deliverable as one comparison table.
SCOREBOARD = []

def record(result):
    """Append a regression_report dict to the scoreboard and show the table."""
    SCOREBOARD.append(result)
    return pd.DataFrame(SCOREBOARD).set_index('model').round(4)


def show_scoreboard():
    return pd.DataFrame(SCOREBOARD).set_index('model').round(4)

print('Helpers defined: regression_report, record, show_scoreboard')

---
# Section 1 — Load the data

Nothing new here conceptually — but note the two given lines after the load, and read the comment attached to them.

In [ ]:
# Load the CSV into `raw`, print its shape and its columns.
raw = pd.read_csv(DATA_PATH)
print('shape:', raw.shape)
print(raw.columns.tolist())

In [ ]:
# Two housekeeping fixes, given to you.
#
# 1. The CSV as published ships a nameless first column that pandas labels
#    'Unnamed: 0'. It is the row number from whoever exported the file -- somebody
#    ran to_csv() without index=False. It is NOT a feature: a flight you are asked
#    to price next week has no row number in your training file. Drop it.
#    (Deliverable 1 never dropped it, which means it was sitting in your X. Section 3
#    measures what that cost you.)
#
# 2. The Deliverable 1 schema table used 'Source City' / 'Price'; the actual CSV
#    uses 'source_city' / 'price'. Documentation drifting from the file is routine.
#    We lowercase everything so the rest of the notebook works either way.

df = raw.copy()
df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
df = df.drop(columns=[c for c in df.columns if c.startswith('unnamed')])

print('shape:', df.shape)
print('columns:', df.columns.tolist())
df.head(3)

---
# Section 2 — EDA that informs a modelling decision

**Goal:** go deeper than Deliverable 1, and tie every observation to a modelling consequence.

Deliverable 1's EDA was descriptive: here is the distribution, here are the categories, here is a scatter plot. Useful, but it stopped short. **EDA earns its keep when it changes what you do next.** So for every plot in this section, the deliverable is not the plot — it is the sentence "...therefore I should \_\_\_ when I model".

Four things we need to know before choosing a model:

| # | What we look at | The modelling decision it drives |
|---|---|---|
| 2a | How `price` is distributed | Which **loss function** (Section 5) and whether to transform the target |
| 2b | Whether relationships are **linear** | Whether a linear model (Section 4) can compete at all |
| 2c | The **booking curve** over `days_left` | Whether a **time-series** model (Section 6) has anything to work with |
| 2d | Category counts and dominance | Which **encoding**, and whether one feature will swamp everything |

---

### 2a — The shape of the target

Deliverable 1 plotted the price histogram and noted it was right-skewed. Go further: plot it **split by class**.

In [ ]:
# 1-2. Price split by class, on a linear and a log x-axis.
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.histplot(data=df, x='price', hue='class', bins=80, element='step', ax=axes[0])
axes[0].set_title('price by class (linear x-axis)')
sns.histplot(data=df, x='price', hue='class', bins=80, element='step',
             log_scale=True, ax=axes[1])
axes[1].set_title('price by class (log x-axis)')
plt.tight_layout()
plt.show()

# 3. Skewness, raw vs log-transformed -- overall and within each class.
print(f"skew  raw price : {df['price'].skew():.3f}")
print(f"skew  log1p     : {np.log1p(df['price']).skew():.3f}")
for cabin, grp in df.groupby('class'):
    print(f"  {cabin:9s} raw skew {grp['price'].skew():.3f} | "
          f"log1p skew {np.log1p(grp['price']).skew():.3f}")

# 4. Count / mean / median / std per class.
df.groupby('class')['price'].agg(['count', 'mean', 'median', 'std']).round(0)

**What you should see, and what it means.** *(3–4 sentences.)*

The distribution is not one skewed hump — it is **two separate populations**, Economy and Business, with Business roughly an order of magnitude more expensive.

That matters for Section 5 in a concrete way. Squared error (MSE) penalises a ₹5,000 miss identically whether the ticket costs ₹6,000 or ₹52,000 — but the first is an 83% error and the second is under 10%. So **a model trained on MSE will quietly prioritise getting Business right**, because that is where the big absolute numbers are.

Write down: the two class means, the raw skew, the log skew, and one sentence on which loss function you now expect to prefer and why.

- **Class means:** Business ₹52,540 (median ₹53,164), Economy ₹6,572 (median ₹5,772), about 8× apart. Two clearly separate humps.
- **Skew:** raw price **1.06**, log1p(price) **0.41**. Inside each class the picture is different: Economy alone has skew 1.70, which the log almost removes (0.04). Business is already symmetric (−0.10), and the log actually makes it *worse* (−1.21). So most of the overall skew comes from mixing two populations, and only the Economy part is a genuine right tail.
- **Loss expectation:** an MSE model will spend its effort on Business, where the big rupee errors are, and be relatively sloppy on Economy, where a ₹2,000 miss is 30%. If the business cares about **percentage** error, I expect MAE (or MSE on log price) to look better on MAPE. If it cares about rupee error on the expensive tickets, MSE is the honest choice. Section 5 tests this.

### 2b — Is this problem linear?

Before you can argue that you need a boosted tree ensemble, you have to know what a straight line gets you. And before you can read a correlation matrix, note the limit of the tool: **Pearson correlation measures only the *linear* part of a relationship.** A perfect U-shaped relationship has a correlation near zero. A low correlation does not mean "no relationship" — it means "no *linear* relationship", which is a much narrower claim.

In [ ]:
num = df[['duration', 'days_left', 'price']]
tmp = num.copy()
tmp['is_business'] = (df['class'] == 'Business').astype(int)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# 1. Numeric columns only.
sns.heatmap(num.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=axes[0])
axes[0].set_title('numeric features only')
# 2. ...plus the strongest categorical as a 0/1 column.
sns.heatmap(tmp.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=axes[1])
axes[1].set_title('with is_business added')
plt.tight_layout()
plt.show()

# 3. Price vs duration with a straight line on top.
s = df.sample(5000, random_state=RANDOM_STATE)
sns.regplot(data=s, x='duration', y='price',
            scatter_kws={'s': 4, 'alpha': 0.3},
            line_kws={'color': 'red'})
plt.title('price vs duration, with a single linear fit (5,000-row sample)')
plt.show()

print(tmp.corr()['price'].round(3))

**Your read.** *(3–4 sentences. Which feature carries the most linear signal? Where does the straight line visibly fail? Does the `duration` correlation being weak prove duration does not matter — and if not, why not?)*

`is_business` carries by far the most linear signal: **r = 0.94** with price, against **0.20** for `duration` and **−0.09** for `days_left`. The straight line through the duration/price scatter visibly fails: the cloud is two horizontal bands (Economy around ₹5–10k, Business around ₹40–70k), and the red line runs through the empty gap between them, so it fits neither band well. The weak `duration` correlation does **not** prove duration is irrelevant. Pearson only measures one global straight line, and duration's effect is conditional (it behaves differently within each class and is tangled up with stops and route) and non-linear, so a low *r* means "no single linear trend", not "no relationship". This tells me a linear model will need `class` above everything else and will leave a lot on the table elsewhere.

### 2c — The booking curve: the one piece of this dataset with a time axis

This is the most important plot in the notebook, and it is the one Deliverable 1 never made.

`days_left` is **days until departure**. It is not a timestamp, but it *is* an ordered horizon — so if you average price at each value of `days_left`, you get a genuine **sequence**: how fare moves as departure approaches. Airlines call this the booking curve, and forecasting it is a real job.

That ordered sequence is the only thing in this dataset a time-series model like ARIMA can operate on. Section 6c comes back to it.

In [ ]:
# 1. The aggregate booking curve. x-axis reversed so time flows toward departure.
curve = df.groupby('days_left')['price'].agg(['mean', 'median', 'count'])
plt.plot(curve.index, curve['mean'], marker='o', ms=3)
plt.gca().invert_xaxis()
plt.xlabel('days until departure  (departure is at the right)')
plt.ylabel('mean price (INR)')
plt.title('booking curve -- all cabins')
plt.show()

# 2. One line per class. Business is ~8x Economy, so give each its own y-axis
#    as well, otherwise Economy's shape is flattened against the bottom.
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for cabin, grp in df.groupby('class'):
    c = grp.groupby('days_left')['price'].mean()
    axes[0].plot(c.index, c.values, marker='o', ms=3, label=cabin)
axes[0].invert_xaxis(); axes[0].legend()
axes[0].set_title('by class, shared y-axis')
axes[0].set_xlabel('days until departure'); axes[0].set_ylabel('mean price (INR)')

ax_e = axes[1]
ax_b = ax_e.twinx()
for cabin, ax, colour in [('Economy', ax_e, 'C1'), ('Business', ax_b, 'C0')]:
    c = df[df['class'] == cabin].groupby('days_left')['price'].mean()
    ax.plot(c.index, c.values, marker='o', ms=3, color=colour, label=cabin)
    ax.set_ylabel(f'{cabin} mean price (INR)', color=colour)
ax_e.invert_xaxis()
ax_e.set_title('by class, each on its own y-axis')
ax_e.set_xlabel('days until departure')
plt.tight_layout()
plt.show()

# 3. Size of the series ARIMA will get in Section 6.
print('points in the booking curve:', len(curve))
print('rows per point: min', curve['count'].min(), '| max', curve['count'].max())
print('mean price at 1, 2, 15, 20, 30, 49 days out:')
print(curve.loc[[1, 2, 15, 20, 30, 49], 'mean'].round(0))

**Read the curve.** *(4–5 sentences.)*

- Is the relationship between `days_left` and price **linear**? At roughly what point does it turn?
- Economy vs Business: same shape, or different?
- This answers something important about Section 4. A linear model given raw `days_left` fits **one straight line** through this curve. Looking at the shape, how badly will that go — and what could you do to a linear model to let it cope? *(Think: what if `days_left` were bucketed, or log-transformed, or squared?)*
- How many data points does the aggregate curve have? Is that a lot or a little for fitting a time-series model?

- **Not linear.** Within each cabin the curve is almost flat from 49 days out to about **16 days out**, then climbs steeply as departure approaches. Economy goes from ~₹4,750 (49 days) to ~₹6,300 (16 days), jumps to ~₹8,400 at 15 days, and reaches **~₹14,600 the day before**, roughly 3× its flat level. Business is flatter in relative terms: ~₹51k to ~₹65k, with almost all of the rise in the last ~10 days.
- **Economy vs Business:** same direction, different shape. Economy has a sharp step around the two-week mark, while Business drifts up gently and then rises in the final days. The all-cabins curve is misleading on its own: it **drops** on the last day (₹30.2k at 2 days, then ₹21.6k at 1 day) only because the Business share of rows falls from 35% to 14%. That is a change in the cabin mix, not a price fall.
- **Linear-model consequence:** one straight line on raw `days_left` will badly over-predict the last two weeks' slope region and under-predict the final days. A linear model could cope if `days_left` were **bucketed** (e.g. 1–3, 4–9, 10–15, 16+), log-transformed (the curve is roughly flat-then-steep, which `log(days_left)` straightens), or interacted with `class`.
- **Size:** the aggregate curve is **49 points** (each the mean of 1,927–6,633 rows). That is very little for a time-series model: ARIMA would have about 40 points to train on and 9 to test, which is barely enough to estimate a handful of parameters.

### 2d — Categories: counts, dominance, and encoding

In [ ]:
# 1. Cardinality and exact spellings of every categorical column.
cats = ['airline', 'source_city', 'departure_time', 'stops',
        'arrival_time', 'destination_city', 'class']
for c in cats:
    print(f'{c:20s} {df[c].nunique():3d}  {sorted(df[c].unique())}')
print(f"\n{'flight':20s} {df['flight'].nunique():3d}  <- near-identifier, kept out of the model")

# 2. Mean price by airline x class -- do all airlines even sell Business?
print('\nmean price by airline x class (NaN = does not sell that cabin):')
display(pd.crosstab(df['airline'], df['class'], values=df['price'],
                    aggfunc='mean').round(0))
print('rows per airline x class:')
display(pd.crosstab(df['airline'], df['class']))

# 3. Cheapest and most expensive routes.
route_price = df.groupby(['source_city', 'destination_city'])['price'].mean().round(0)
print('cheapest 5 routes:')
display(route_price.sort_values().head(5))
print('most expensive 5 routes:')
display(route_price.sort_values().tail(5))

> **Which columns are ordered, and which are not?** This matters in Section 4 more than you would guess.
>
> - `stops` is **ordinal**: `zero < one < two_or_more`. That is a real ranking.
> - `departure_time` / `arrival_time` are **ordinal**: `Early_Morning < Morning < Afternoon < Evening < Night < Late_Night` — they are a clock.
> - `airline`, `source_city`, `destination_city` are **nominal**. There is no sense in which Vistara > SpiceJet.
>
> Deliverable 1 used `LabelEncoder` on all seven, which assigns integers **alphabetically**. For `stops` that gives `one=0, two_or_more=1, zero=2` — telling the model a direct flight ranks *above* a two-stop flight. Not just arbitrary: backwards.
>
> A tree can route around this by spending extra splits to separate the categories again, so the cost shows up as wasted capacity. **A linear model cannot route around it at all** — it gets one coefficient per column and is forced to assume evenly-spaced, ordered effects. Section 4 shows you exactly how much that costs.

### 2e — EDA summary: four decisions

Fill this in before you model. This is the deliverable for Section 2 — not the plots.

| Observation | Therefore, when I model, I will... |
|---|---|
| Target distribution is... **two populations** (Economy mean ₹6.6k, Business ₹52.5k), raw skew 1.06. The real right tail is inside Economy (skew 1.70); Business is symmetric. | ...compare MSE vs MAE vs quantile losses explicitly and report **MAPE alongside MAE**, because an MSE fit will prioritise Business. I'll keep the raw target for comparability, but note log(price) as an option because it fixes Economy's skew and not Business's. |
| The strongest single predictor appears to be... **`class`** (r = 0.94 as a 0/1 column; worth ~0.98 vs ~0.19 R² with/without). | ...assert that `class` is in the `ColumnTransformer` (coverage check), expect it to dominate importances and coefficients, and judge every other feature by what it adds *after* class. |
| The `days_left` relationship is... **flat from 49 to ~16 days, then steeply rising** toward departure (Economy ~3×), with a cabin-mix artefact in the aggregate. | ...expect trees to beat a linear model on raw `days_left` (a linear model would need buckets/log); for ARIMA, accept that it gets only 49 aggregate points and is confounded by cabin mix. |
| Ordered categories are... **`stops` and the two time-of-day columns**. `airline`/cities are nominal. Note: raw mean price is *not* monotone in stops (zero ₹9.4k, two_or_more ₹14.1k, one ₹22.9k) because of class/route mix. | ...one-hot the nominals, `OrdinalEncoder` with explicit orders (`zero<one<two_or_more`, Early_Morning→Late_Night) for the ordinals, never `LabelEncoder`'s alphabetical order. For a linear model, remember a single coefficient forces even spacing. |

---
# Section 3 — Data leakage

**Goal:** find the three places the Deliverable 1 setup let information leak, fix each one, and **measure what each was actually worth** rather than assuming.

> **Leakage** is when information reaches the model at training time that will not be available at prediction time.

It is not exotic. It is the default result of writing code in the order that feels natural — "clean everything, then split" — because every transform fitted before the split has already seen the test set.

Deliverable 1 had three leaks:

| # | Leak | Where |
|---|---|---|
| 1 | **The row counter.** `Unnamed: 0` was never dropped, so it sat in `X`. A future flight has no row number. | D1 §3a |
| 2 | **Duplicate rows across the split.** Once the unique counter is removed, this dataset contains repeated rows; copies can land in train *and* test, so the model is graded on rows it memorised. | D1 §3c |
| 3 | **Encoders fitted on the full dataset.** `df[col] = le.fit_transform(df[col])` ran in §3b, *before* the split in §3c. | D1 §3b |

**Be precise about leak 3's severity**, because overstating it is its own error. A plain `LabelEncoder` on complete categorical columns learns a list of category *names* — it learns nothing about price, so the practical damage is small. It is still **wrong as a habit**, because the identical code shape is severe the moment the transform learns something numeric:

- a `StandardScaler` fitted pre-split leaks the test set's mean and variance
- a **target encoder** fitted pre-split leaks the test set's *prices* directly
- an imputer fitted pre-split leaks the test distribution

You will not remember which transforms are safe at 2am. **Fit every transform inside the split, always.** That is what `Pipeline` is for.

### 3a — Reproduce the Deliverable 1 baseline (given)

Given to you complete, so every member starts from the identical reference number. Read it and match each line to a section of the old notebook.

In [ ]:
def deliverable1_baseline(path, random_state=RANDOM_STATE):
    """Reproduce Deliverable 1 exactly -- leaks and all.

      D1 S1  : read_csv
      D1 S2a : dropna
      D1 S3a : drop 'flight'        <- but NOT the row counter
      D1 S3b : LabelEncoder on the FULL frame, before the split   <- leak 3
      D1 S3c : X = everything except price; 80/20 split           <- dupes straddle it
      D1 S4  : XGBRegressor, same hyperparameters
    """
    from sklearn.preprocessing import LabelEncoder

    frame = pd.read_csv(path)
    frame.columns = [c.strip().lower().replace(' ', '_') for c in frame.columns]
    frame = frame.dropna().drop(columns=['flight'])

    for col in ['airline', 'source_city', 'departure_time', 'stops',
                'arrival_time', 'destination_city', 'class']:
        # Fitted on 100% of rows -- including the ones we will call "unseen".
        frame[col] = LabelEncoder().fit_transform(frame[col])

    X = frame.drop(columns=['price'])
    y = frame['price']
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=random_state)

    model = XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.1,
                         subsample=0.8, colsample_bytree=0.8,
                         random_state=random_state, n_jobs=-1)
    model.fit(X_tr, y_tr)
    return model, X_tr, X_te, y_tr, y_te, model.predict(X_te)


t0 = time.time()
d1_model, d1_X_tr, d1_X_te, d1_y_tr, d1_y_te, d1_pred = deliverable1_baseline(DATA_PATH)
print(f'trained in {time.time() - t0:.1f}s')
print('features the D1 model was given:', d1_X_tr.columns.tolist())

record(regression_report(d1_y_te, d1_pred, 'D1 baseline (3 leaks)'))

In [ ]:
# Where did the row counter rank in the D1 model?
imp = pd.Series(d1_model.feature_importances_, index=d1_X_tr.columns) \
        .sort_values(ascending=False)
print('D1 gain importances:')
print(imp.round(4))
print(f"\nrow counter rank: {imp.index.get_loc('unnamed:_0') + 1} of {len(imp)}")

# Why does a row number carry signal at all? Because the file is sorted.
raw_sorted = pd.read_csv(DATA_PATH)
raw_sorted.columns = [c.strip().lower().replace(' ', '_')
                      for c in raw_sorted.columns]
raw_sorted.groupby('class')['unnamed:_0'].agg(['min', 'max', 'count'])

### 3b — Remove the leaks, one at a time

One change per step, so each result is **attributable**. Change three things at once and you learn nothing about any of them.

In [ ]:
# Step 1 -- the D1 baseline with the row counter dropped.
# Same body as deliverable1_baseline; the only change is dropping 'unnamed:_0'.
# `dedupe` is switched on in Step 2, so both steps share one code path and differ
# only in the one thing being measured.
from sklearn.preprocessing import LabelEncoder

def d1_without_counter(path, dedupe=False, random_state=RANDOM_STATE):
    frame = pd.read_csv(path)
    frame.columns = [c.strip().lower().replace(' ', '_') for c in frame.columns]
    frame = frame.dropna().drop(columns=['flight', 'unnamed:_0'])   # <- fix 1

    if dedupe:                                                        # <- fix 2
        before = len(frame)
        frame = frame.drop_duplicates()
        print('dropped', before - len(frame), 'duplicate rows')

    for col in ['airline', 'source_city', 'departure_time', 'stops',
                'arrival_time', 'destination_city', 'class']:
        frame[col] = LabelEncoder().fit_transform(frame[col])    # leak 3 still present

    X = frame.drop(columns=['price'])
    y = frame['price']
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=random_state)

    model = XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.1,
                         subsample=0.8, colsample_bytree=0.8,
                         random_state=random_state, n_jobs=-1)
    model.fit(X_tr, y_tr)
    return y_te, model.predict(X_te)


y_te, pred = d1_without_counter(DATA_PATH)
record(regression_report(y_te, pred, 'no row counter'))

In [ ]:
# Step 2 -- also remove duplicate rows, BEFORE the split.
# Dedupe happens inside d1_without_counter after the counter and 'flight' are gone,
# i.e. on exactly the columns the model sees -- so no identical (X, y) row can sit
# on both sides of the split.
y_te, pred = d1_without_counter(DATA_PATH, dedupe=True)
record(regression_report(y_te, pred, 'no counter + deduped'))

### 3c — Step 3: a `Pipeline`, so leakage becomes structurally impossible

A **`Pipeline`** is a list of `(name, transform)` steps ending in a model, wrapped to behave as one estimator. What matters is what `.fit()` does:

```
pipe.fit(X_train, y_train)
   -> every transform learns ONLY from X_train
pipe.predict(X_test)
   -> every transform APPLIES what it learned; it never re-learns from X_test
```

You cannot accidentally fit on the test set, because you never call `.fit` on a transform yourself. And this becomes essential rather than merely tidy in Section 4, where we cross-validate over many `alpha` values: without a pipeline, the scaler would be refitted on each fold's test data and every number would be slightly wrong.

A **`ColumnTransformer`** routes different columns to different transforms:

```python
ColumnTransformer([
    ('nom', OneHotEncoder(handle_unknown='ignore'), nominal_cols),
    ('ord', OrdinalEncoder(categories=[...]),       ordinal_cols),
    ('num', StandardScaler(),                       numeric_cols),
])
```

`handle_unknown='ignore'` is the production-mindset flag: if an unseen category appears at prediction time (a new airline launches), encode it as all-zeros instead of crashing.

In [ ]:
# Exact spellings taken from the Section 2d printout.
NOMINAL = ['airline', 'source_city', 'destination_city', 'class']
ORDINAL = ['stops', 'departure_time', 'arrival_time']
NUMERIC = ['duration', 'days_left']

STOPS_ORDER = ['zero', 'one', 'two_or_more']
TIME_ORDER  = ['Early_Morning', 'Morning', 'Afternoon', 'Evening', 'Night', 'Late_Night']

# Fail loudly if an order list does not match the data.
assert set(STOPS_ORDER) == set(df['stops'].unique())
for col in ['departure_time', 'arrival_time']:
    assert set(TIME_ORDER) == set(df[col].unique()), col
print('category orders match the data')

> **⚠️ `remainder='drop'` is a footgun, and this is the moment to respect it.** The `ColumnTransformer` you are about to build **silently discards** every column you did not put in a group. No error. No warning.
>
> On this dataset that mistake is expensive in a very specific way: `class` is the strongest single predictor here, worth roughly **0.98 R² with it versus 0.19 without**. Forget it and your model is near-useless — and nothing in the output will say so. You will just see a bad score and start blaming your hyperparameters.
>
> So do not verify coverage by eye. Make the computer do it.

In [ ]:
# Prove every feature column is accounted for BEFORE building the pipeline.
# `clean` is built in the next-but-one cell as df.drop_duplicates(), so it has
# exactly df's columns -- checking df here is the same check, one cell earlier.
assigned = set(NOMINAL) | set(ORDINAL) | set(NUMERIC)
features = set(df.columns) - {'price'}
dropped  = features - assigned

print('assigned to a group :', sorted(assigned))
print('will be DROPPED     :', sorted(dropped))

# 'flight' SHOULD be dropped -- the high-cardinality near-identifier. Anything else is a bug.
assert dropped == {'flight'}, f'unexpectedly dropping: {sorted(dropped - {"flight"})}'
print('coverage OK')

In [ ]:
# Trees: no scaling (a split threshold does not care about units).
prep_tree = ColumnTransformer([
    ('nom', OneHotEncoder(handle_unknown='ignore'), NOMINAL),
    ('ord', OrdinalEncoder(categories=[STOPS_ORDER, TIME_ORDER, TIME_ORDER],
                           handle_unknown='use_encoded_value',
                           unknown_value=-1), ORDINAL),
    ('num', 'passthrough', NUMERIC),
], remainder='drop')

# Linear: identical, except numerics are standardised (mandatory for Lasso/Ridge).
prep_linear = ColumnTransformer([
    ('nom', OneHotEncoder(handle_unknown='ignore'), NOMINAL),
    ('ord', OrdinalEncoder(categories=[STOPS_ORDER, TIME_ORDER, TIME_ORDER],
                           handle_unknown='use_encoded_value',
                           unknown_value=-1), ORDINAL),
    ('num', StandardScaler(), NUMERIC),
], remainder='drop')

prep_tree

In [ ]:
# The clean dataset and the ONE split every model in Sections 4-6 shares.
clean = df.drop_duplicates().reset_index(drop=True)
print(f'dropped {len(df) - len(clean):,} exact duplicate rows (flight number included)')

X = clean.drop(columns=['price'])
y = clean['price']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE)
print(f'{len(X_train):,} train / {len(X_test):,} test')

# Honest baseline: same XGBoost as D1, but every transform is fitted inside the pipeline.
xgb_pipe = Pipeline([('prep', prep_tree),
                     ('model', XGBRegressor(n_estimators=300, max_depth=6,
                                            learning_rate=0.1, subsample=0.8,
                                            colsample_bytree=0.8,
                                            random_state=RANDOM_STATE,
                                            n_jobs=-1))])
xgb_pipe.fit(X_train, y_train)
print('features after encoding:', len(xgb_pipe.named_steps['prep'].get_feature_names_out()))
record(regression_report(y_test, xgb_pipe.predict(X_test), 'XGBoost (no leaks)'))

### 3d — Target leakage: the expensive kind

The three leaks above were about *process*. This one is about a *feature*, and it is the one that ends projects.

Here is a feature a well-meaning teammate will propose:

```python
df['price_per_hour'] = df['price'] / df['duration']
```

It is intuitive, it is a real industry unit (yield per block hour), and adding it will take your R² to something like 0.999.

**It is computed from the target.** At prediction time you do not have `price` — predicting it is the whole job. The feature can only exist for rows where you already know the answer.

**Target leakage** is the most expensive mistake in applied ML precisely because it does not look like a bug. Nothing crashes, no warning fires, your metrics improve, and the model is worthless.

> **The test, five seconds per feature:** *"at the moment I need a prediction, do I have this value?"* If the honest answer is "no, I would need the price first" — delete it.

In [ ]:
# Target leakage, demonstrated -- and NOT recorded on the scoreboard.
leaky = clean.copy()
leaky['price_per_hour'] = leaky['price'] / leaky['duration']
Xl = leaky[['price_per_hour', 'duration', 'days_left']]
yl = leaky['price']
Xl_tr, Xl_te, yl_tr, yl_te = train_test_split(Xl, yl, test_size=0.2,
                                              random_state=RANDOM_STATE)
m = XGBRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
m.fit(Xl_tr, yl_tr)
print(regression_report(yl_te, m.predict(Xl_te), 'TARGET LEAK -- do not ship'))

del leaky, Xl, yl, Xl_tr, Xl_te, yl_tr, yl_te, m    # deleted, as promised

### 3e — One split is an anecdote

Even leak-free, our score rests on **one** arbitrary 80/20 split. Change `random_state` and the number moves. How much is measurable, and you should measure it before quoting anything.

**K-fold cross-validation** splits the data into k parts, trains k times, validates on each part once. The **mean** is a better estimate; the **standard deviation** tells you how much your single number could have been luck.

> Quote scores as **mean ± std**. A model at 0.95 ± 0.002 and one at 0.95 ± 0.04 are not the same model, and the second is not ready to ship.

In [ ]:
# 5-fold CV on the honest XGBoost pipeline -- full data, no subsampling.
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
t0 = time.time()
scores = cross_validate(xgb_pipe, X, y, cv=cv,
                        scoring=['r2', 'neg_mean_absolute_error'],
                        n_jobs=1, return_train_score=True)
print(f'5 folds in {time.time() - t0:.0f}s\n')

cv_table = pd.DataFrame({
    'train_r2': scores['train_r2'],
    'test_r2':  scores['test_r2'],
    'train_MAE': -scores['train_neg_mean_absolute_error'],
    'test_MAE':  -scores['test_neg_mean_absolute_error'],
}, index=[f'fold {i+1}' for i in range(5)])
display(cv_table.round(4))

for col in cv_table:
    s = cv_table[col]
    print(f'{col:10s} mean {s.mean():10.4f} ± {s.std():.4f}   (min {s.min():.4f}, max {s.max():.4f})')

CV_R2_MEAN, CV_R2_STD = cv_table['test_r2'].mean(), cv_table['test_r2'].std()
CV_MAE_MEAN, CV_MAE_STD = cv_table['test_MAE'].mean(), cv_table['test_MAE'].std()
print(f'\ntrain - test R2 gap: {cv_table["train_r2"].mean() - CV_R2_MEAN:.4f}')

**The leak ledger.** *(4–6 sentences.)*

- Which single fix cost the most R²? Why that one?
- Did your honest number end up lower than your Deliverable 1 number, higher, or about the same? **Whichever it is, which number would you put in a report to American Airlines, and how would you explain the difference to someone who already saw the first one?** Note that "the leak was worth almost nothing here" is a perfectly good answer — if that is what you measured, say it and explain why.
- What does your CV standard deviation tell you about how much to trust one decimal place of these scores?

- **The row counter cost the most:** removing `unnamed:_0` took R² from **0.9782 → 0.9739** and MAE from **₹1,911 → ₹2,100** (+₹190). It ranked **2nd of 10** in gain importance, behind only `class`. It carries signal because the file is sorted: Economy is rows 0–206,665 and Business 206,666–300,152, and within a class neighbouring rows are the same airline/route at adjacent `days_left`. So a test row's row number places it among near-identical training rows. **Deduplication barely changed anything:** 2,213 duplicates (0.7% of rows) gave R² 0.9741 and MAE ₹2,107. They are ordinary fares the model predicted well anyway. **The pipeline** took R² *up* to 0.9761 and MAE to ₹1,987. That gain comes from better **encoding** (one-hot airline/cities, correctly ordered stops/times), not from fixing leak 3, since a `LabelEncoder` fitted on the full data learns only category names.
- **Net effect:** the honest 5-fold CV number is **R² 0.9765 ± 0.0005, MAE ₹1,975 ± 20**, only ~0.002 R² / ~₹65 worse than D1's 0.9782 / ₹1,911. Here the leaks were worth little. The number I'd report is the CV one. To someone who saw 0.978 I'd say: "the old number was graded partly on information a real booking wouldn't have; the honest number is 0.977, and the gap is small, which is itself good news."
- One caveat I found: `clean = df.drop_duplicates()` dropped **0 rows**, because `flight` makes every row unique. The 2,213 rows that are duplicates in every *model-visible* column (identical X and price, different flight number) are still in the honest split. That is small, but not zero.
- **CV std of 0.0005 R² (₹20 MAE)** means the third decimal of R² is trustworthy and the fourth is noise. Any two models closer than ~0.001 R² or ~₹40 MAE on one split should be called a tie.

---
# Section 4 — Lasso vs Ridge: two ways to regularise a linear model

**Goal:** understand what regularisation *is*, what the L1 and L2 penalties do differently, and why that difference is useful rather than academic.

### Why fit a linear model at all, when you already have XGBoost?

Three reasons, and none of them is "it might win":

1. **It is a reference point.** "XGBoost got R² 0.95" is meaningless until you know a linear model gets 0.90 — or 0.55.
2. **It is interpretable.** A linear model hands you one coefficient per feature: *this airline adds ₹1,800 to the fare, all else equal.* No tree ensemble gives you a sentence like that, and that sentence is often what a client actually wants.
3. **Lasso does feature selection for free**, which is the real payoff in this section.

### The shared objective

Both models minimise squared error **plus** a penalty on the size of the coefficients:

```
Ridge  (L2):   minimise   ||y - Xw||²  +  α · Σ wⱼ²
Lasso  (L1):   minimise   ||y - Xw||²  +  α · Σ |wⱼ|
```

The only difference is `wⱼ²` versus `|wⱼ|`. That one change produces completely different behaviour:

| | **Ridge (L2)** | **Lasso (L1)** |
|---|---|---|
| Penalty | sum of **squared** coefficients | sum of **absolute** coefficients |
| Shrinks coefficients | yes, smoothly toward zero | yes |
| Sets them to **exactly zero** | essentially never | **yes** — that is the point |
| Correlated features | **shares** the weight between them | **picks one**, zeroes the rest |
| Gives you | a stable model using every feature | a **sparse** model, i.e. feature selection |
| Closed-form solution | yes | no — needs iterative optimisation |
| Reach for it when | you believe most features matter a little | you believe few features matter a lot |

### Why does L1 produce exact zeros and L2 does not?

The geometric picture is the one worth remembering. Think of the penalty as a budget constraining where `w` can live:

- The **L2** budget `Σwⱼ² ≤ t` is a **circle** (sphere). Smooth, no corners.
- The **L1** budget `Σ|wⱼ| ≤ t` is a **diamond**, and its corners sit **on the axes**.

The solution lands where the error contours first touch that region. A diamond's corners stick out, so contours tend to touch at a corner — and a corner *is* a point where some coefficient equals exactly zero. A circle has no corners, so the touch point almost always has every coefficient slightly non-zero.

> **⚠️ Scaling is mandatory here, not optional.** Both penalties sum over coefficient *magnitudes*, and a coefficient's magnitude depends on its feature's units. `duration` in hours (0–50) and `days_left` in days (1–49) are comparable, but if one were in minutes its coefficient would be ~60× smaller and the penalty would barely touch it. **Unscaled features mean the penalty silently punishes some features more than others.** This is why `prep_linear` has a `StandardScaler` and `prep_tree` does not — and why both live inside a `Pipeline`, so the scaler is fitted per fold.

In [ ]:
# Three linear models through `prep_linear`. Fit times kept for the 6c table.
models = {
    'LinearRegression (no penalty)': LinearRegression(),
    'Ridge (alpha=1)': Ridge(alpha=1.0, random_state=RANDOM_STATE),
    'Lasso (alpha=1)': Lasso(alpha=1.0, random_state=RANDOM_STATE, max_iter=5000),
}
for name, est in models.items():
    pipe = Pipeline([('prep', prep_linear), ('model', est)])
    t0 = time.time()
    pipe.fit(X_train, y_train)
    rep = regression_report(y_test, pipe.predict(X_test), name)
    rep['fit_seconds'] = round(time.time() - t0, 1)
    tbl = record(rep)
tbl

### 4a — What `alpha` actually does

`alpha` is the **regularisation strength** — how much you charge the model for large coefficients.

- `alpha → 0`: no penalty. The model fits the training data as hard as it can. **High variance**, overfits.
- `alpha → ∞`: the penalty dominates. All coefficients are crushed toward zero and the model predicts roughly a constant. **High bias**, underfits.

Somewhere in between is the right value, and you find it by validation, not by taste. This is the **bias–variance tradeoff** with a dial on it.

In [ ]:
# Sweep alpha for both models, train and test R2.
import warnings
from sklearn.exceptions import ConvergenceWarning

alphas = np.logspace(-3, 3, 13)      # 0.001 ... 1000, log-spaced
rows = []
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always', ConvergenceWarning)
    for a in alphas:
        for name, est in [('Ridge', Ridge(alpha=a)),
                          ('Lasso', Lasso(alpha=a, max_iter=5000))]:
            pipe = Pipeline([('prep', prep_linear), ('model', est)])
            pipe.fit(X_train, y_train)
            rows.append({'alpha': a, 'model': name,
                         'train_r2': pipe.score(X_train, y_train),
                         'test_r2': pipe.score(X_test, y_test),
                         'n_zero': int((pipe.named_steps['model'].coef_ == 0).sum())})
n_conv = sum(issubclass(w.category, ConvergenceWarning) for w in caught)
print(f'ConvergenceWarnings during the sweep: {n_conv}')
sweep = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for name, grp in sweep.groupby('model'):
    for ax in axes:
        ax.semilogx(grp['alpha'], grp['test_r2'], marker='o', label=f'{name} test')
        ax.semilogx(grp['alpha'], grp['train_r2'], ls='--', alpha=0.5,
                    label=f'{name} train')
for ax in axes:
    ax.set_xlabel('alpha (regularisation strength)'); ax.set_ylabel('R2'); ax.legend()
axes[0].set_title('full range')
axes[1].set_title('zoomed to the top of the curve')
axes[1].set_ylim(sweep['test_r2'].max() - 0.01, sweep['test_r2'].max() + 0.002)
plt.tight_layout()
plt.show()

best = sweep.loc[sweep.groupby('model')['test_r2'].idxmax()]
print('best alpha per model (by held-out R2):')
display(best.round(5))
display(sweep.pivot(index='alpha', columns='model', values='test_r2').round(5))

In [ ]:
# Which features did Lasso switch off?
lasso_pipe = Pipeline([('prep', prep_linear),
                       ('model', Lasso(alpha=100, max_iter=20000))])
lasso_pipe.fit(X_train, y_train)

names = lasso_pipe.named_steps['prep'].get_feature_names_out()
coefs = pd.Series(lasso_pipe.named_steps['model'].coef_, index=names)

print('Lasso alpha=100')
print('  features kept:  ', (coefs != 0).sum())
print('  features zeroed:', (coefs == 0).sum(), '->', coefs[coefs == 0].index.tolist())
print(f"  test R2 {lasso_pipe.score(X_test, y_test):.4f} | "
      f"MAE {mean_absolute_error(y_test, lasso_pipe.predict(X_test)):,.0f}")

# Same alpha, Ridge: count the exact zeros.
ridge_pipe = Pipeline([('prep', prep_linear), ('model', Ridge(alpha=100))])
ridge_pipe.fit(X_train, y_train)
ridge_coefs = pd.Series(ridge_pipe.named_steps['model'].coef_, index=names)
print('\nRidge alpha=100')
print('  exact zeros:', (ridge_coefs == 0).sum(),
      f'| smallest |coef|: {ridge_coefs.abs().min():.2f}')

# Watch the zero count climb with alpha.
print('\nLasso zeros as alpha grows:')
for a in [1, 10, 30, 100, 300, 1000, 3000]:
    m = Pipeline([('prep', prep_linear), ('model', Lasso(alpha=a, max_iter=20000))])
    m.fit(X_train, y_train)
    print(f'  alpha={a:5d}  zeroed {(m.named_steps["model"].coef_ == 0).sum():2d} / {len(names)}'
          f'   test R2 {m.score(X_test, y_test):.4f}')

print('\nLargest surviving coefficients (INR per unit; numerics are per 1 std):')
side = pd.DataFrame({'lasso_100': coefs, 'ridge_100': ridge_coefs})
side.loc[coefs[coefs != 0].sort_values(key=abs, ascending=False).head(15).index].round(0)

In [ ]:
# ElasticNet -- both penalties at once.
rows = []
for r in [0.1, 0.5, 0.9]:
    pipe = Pipeline([('prep', prep_linear),
                     ('model', ElasticNet(alpha=1.0, l1_ratio=r, max_iter=5000))])
    pipe.fit(X_train, y_train)
    rep = regression_report(y_test, pipe.predict(X_test), f'ElasticNet l1_ratio={r}')
    rep['n_zero'] = int((pipe.named_steps['model'].coef_ == 0).sum())
    rows.append(rep)
# Reference rows from the scoreboard, for comparison.
ref = [r for r in SCOREBOARD if r['model'] in ('Ridge (alpha=1)', 'Lasso (alpha=1)')]
pd.DataFrame(ref + rows).set_index('model')[['MAE', 'RMSE', 'R2', 'MAPE%', 'n_zero']].round(4)

**Lasso vs Ridge: your verdict.** *(5–7 sentences.)*

1. Which linear model scored best, and at what alpha?
2. How many features did Lasso zero out? Name two it discarded and say whether that was reasonable given your Section 2 EDA.
3. Did `LinearRegression` with no penalty do noticeably worse than the regularised versions? With ~300k rows and only ~30 one-hot columns, you have far more data than parameters — explain why that makes regularisation matter *less* here than it would on a wide, small dataset. **This is the honest answer, and it is more useful than pretending regularisation saved you.**
4. If a PM asked for "a model I can explain to a non-technical stakeholder", which of your models so far would you hand over, and what would you lose by doing so?

1. **Best linear model:** Lasso at **alpha = 1**, test R² **0.90961**. Ridge was flat at **0.90960** across the whole sweep (0.001 to 1000), and plain LinearRegression scored 0.90960. The differences are in the fifth decimal, so effectively a three-way tie at R² ≈ 0.910, MAE ≈ ₹4,500. (Lasso at alpha = 100 has a slightly *better* MAE, ₹4,406, at R² 0.9078, so it trades a little squared error for absolute error.)
2. **Lasso zeroed 9 of 25 features at alpha = 100** (and 1 at alpha = 1, 18 at alpha = 300): the `GO_FIRST`, `Indigo` and `SpiceJet` airline dummies, plus `Bangalore`, `Chennai` and `Mumbai` as both source and destination. That fits the EDA. 2d showed those three airlines sell only Economy at very similar means (₹5.3–6.2k), so once `class` is in the model they are interchangeable, and Lasso folds them into a shared baseline (possible because every one-hot level is kept). The route spread was narrow (₹17k–25k means, mostly cabin mix), so it is reasonable for several city dummies to go. Ridge at the same alpha zeroed **exactly 0** (smallest |coef| ≈ 40). That is the L1/L2 difference, measured.
3. **No, LinearRegression did no worse.** With 240k training rows and only 25 encoded columns (~10,000 rows per parameter), the unpenalised fit already has negligible variance: train R² 0.9097 vs test 0.9096. Regularisation reduces variance, and there was none to remove. The linear model's real problem is **bias**: it can't bend to the booking curve or the class-specific effects, which is why its MAPE is ~42%. On a wide, small dataset (more columns than rows) the penalty would matter a lot. Here it barely does.
4. For a non-technical stakeholder I'd hand over the **Lasso (alpha = 100)**: 16 coefficients that read as sentences. Business adds **₹44,585**; each step up in stops adds **₹4,919**; Vistara adds **₹2,554**; and each extra ~2 weeks (one std, 13.6 days) of booking lead time knocks off **₹1,668**. What I'd lose is accuracy: MAE ~₹4,400 vs ~₹1,100–2,000 for the trees, and a 42% MAPE means Economy quotes can be off by nearly half. It explains prices well and predicts them badly.

*ElasticNet note:* at alpha = 1 every blend was much worse (R² 0.52 / 0.67 / 0.88 for l1_ratio 0.1 / 0.5 / 0.9). That is not because blending is bad. scikit-learn's ElasticNet divides the error term by 2·n, so its L2 part at alpha = 1 is roughly a Ridge with alpha ≈ n·(1 − l1_ratio), i.e. hundreds of thousands. Same alpha, different scale. It would need alpha around 0.001–0.01 to compete.

---
# Section 5 — Loss functions: what are you actually asking the model to minimise?

**Goal:** understand that "train a model" means "minimise a specific definition of wrong", that you *choose* that definition, and that the default is not always right.

Every model in Sections 3 and 4 minimised **squared error**, because that is the default and nobody told you otherwise. That was a choice, made silently, and it had consequences you can measure.

### Loss vs metric — keep these straight

| | **Loss function** | **Evaluation metric** |
|---|---|---|
| Used | *during* training, by the optimiser | *after* training, by you |
| Must be | differentiable (for gradient methods) | anything computable |
| Purpose | tells the model which direction to move | tells you how good the result is |
| Example | `reg:squarederror` | R², MAPE |

They do not have to match — but if your loss and your metric disagree about what "wrong" means, you are optimising for one thing and grading on another. That is worth doing deliberately, never by accident.

### The four losses that matter for regression

Let the error be `e = ŷ - y`.

| Loss | Formula | Penalises | Predicts the | Outlier behaviour |
|---|---|---|---|---|
| **Squared / L2 (MSE)** | `e²` | large errors **quadratically** | conditional **mean** | very sensitive — one bad row can dominate |
| **Absolute / L1 (MAE)** | `|e|` | errors **linearly** | conditional **median** | robust |
| **Huber** | `e²` for small `|e|`, linear beyond `δ` | both, by region | something between | robust, and smooth at zero |
| **Quantile / pinball** | `α·e` if `e<0`, `(α−1)·e` if `e>0` | over- and under-prediction **differently** | the **α-th quantile** | tunable, asymmetric |

Two consequences worth internalising:

- **MSE predicts the mean; MAE predicts the median.** That is not a detail. On the right-skewed price distribution you plotted in Section 2a, the mean sits well above the median, so these two losses produce *systematically different* predictions on the same data.
- **Huber exists because MSE and MAE each have one flaw.** MSE overreacts to outliers; MAE has a kink at zero that makes optimisation less stable and treats a ₹1 error with the same gradient as a ₹10,000 one. Huber is quadratic near zero and linear in the tails — and `δ` is where it switches.

> ### ⚠️ ReLU is an activation function, not a loss function
>
> This confusion is extremely common and worth clearing up precisely, because the two things live in different places in a model.
>
> **ReLU** is `f(x) = max(0, x)`. It is an **activation function**: it sits *inside* a neural network, transforming one layer's output before it reaches the next. It takes one number and returns one number. It has no idea what the true label is, so **it cannot be a loss** — a loss must compare a prediction to a truth.
>
> ```
> input -> [linear layer] -> ReLU -> [linear layer] -> ReLU -> output -> LOSS(output, truth)
>                             ^^^^                     ^^^^              ^^^^
>                        activations: shape the model   |          loss: scores the model
> ```
>
> **But `max(0, ·)` genuinely does appear inside loss functions**, which is probably where the confusion comes from — and this part is useful:
>
> - **Hinge loss** (classification, SVMs): `max(0, 1 − y·ŷ)`. Zero penalty once you are correct by a margin, linear penalty otherwise. That is literally a ReLU applied to the margin.
> - **Pinball / quantile loss** (regression, below): `max(α·e, (α−1)·e)` — two ReLU-shaped arms joined at zero, with **different slopes** on each side. The asymmetry of those slopes is the whole mechanism.
>
> So: ReLU is a **building block** of asymmetric and margin-based losses, not a loss itself. If a teammate says "let's use ReLU as our loss", the useful reply is "do you mean a hinge or a quantile loss?" — because that is almost always the idea underneath.

### 5a — Quantile loss, and why it is the interesting one for pricing

Every loss above is **symmetric**: over-predicting by ₹500 costs exactly what under-predicting by ₹500 costs.

**Pricing is not symmetric.** Consider the two ways to be wrong:

- **Quote too low.** You sell the seat, but you left money on the table — a loss of margin on one booking.
- **Quote too high.** The customer books a competitor. You lose the entire sale, and possibly the customer.

Those costs are not equal, and which is worse depends on the business situation — a nearly-full flight versus one departing half-empty. A symmetric loss cannot express any of that. **Quantile loss can**, because `α` sets the asymmetry:

- `α = 0.5` → symmetric → predicts the **median** (this is exactly MAE)
- `α = 0.9` → under-prediction punished 9× harder → predicts the **90th percentile**, a deliberately high quote
- `α = 0.1` → over-prediction punished 9× harder → predicts the **10th percentile**, a deliberately low quote

Fit three of them and you get a **prediction interval** rather than a single number — "this fare is most likely ₹6,400, and we are fairly confident it is between ₹4,900 and ₹8,800". For a pricing recommendation, that interval is far more useful than a point estimate, because it tells the user how much to trust it.

In [ ]:
print('xgboost', xgboost.__version__, '-> reg:absoluteerror available (needs >= 1.7)')

models = {
    'MSE  (reg:squarederror)':  XGBRegressor(objective='reg:squarederror',
                                    n_estimators=300, max_depth=6, learning_rate=0.1,
                                    random_state=RANDOM_STATE, n_jobs=-1),
    'MAE  (reg:absoluteerror)': XGBRegressor(objective='reg:absoluteerror',
                                    n_estimators=300, max_depth=6, learning_rate=0.1,
                                    random_state=RANDOM_STATE, n_jobs=-1),
    'Huber (sklearn GBR)':      GradientBoostingRegressor(loss='huber', alpha=0.9,
                                    n_estimators=200, max_depth=5,
                                    random_state=RANDOM_STATE),
}
loss_preds = {}
for name, est in models.items():
    pipe = Pipeline([('prep', prep_tree), ('model', est)])
    t0 = time.time()
    pipe.fit(X_train, y_train)
    loss_preds[name] = pipe.predict(X_test)
    rep = regression_report(y_test, loss_preds[name], name)
    rep['fit_seconds'] = round(time.time() - t0, 1)
    tbl = record(rep)
display(tbl.loc[list(models)])

# Mean vs median, measured: the MSE model should sit above the MAE model on average.
for name, p in loss_preds.items():
    print(f'{name:26s} mean prediction {p.mean():9,.0f}  | mean residual (pred - true) {np.mean(p - y_test):8,.0f}')

# The units lesson: XGBoost pseudo-Huber with the default huber_slope=1, then at
# 1,000 and 10,000 INR -- the scale of the errors we actually make.
# Shown, not recorded -- the first is a known-misconfigured model.
print()
for slope in [1.0, 1000.0, 10000.0]:
    m = Pipeline([('prep', prep_tree),
                  ('model', XGBRegressor(objective='reg:pseudohubererror',
                                         huber_slope=slope, n_estimators=300,
                                         max_depth=6, learning_rate=0.1,
                                         random_state=RANDOM_STATE, n_jobs=-1))])
    m.fit(X_train, y_train)
    r = regression_report(y_test, m.predict(X_test), f'pseudo-Huber, huber_slope={slope:g}')
    print(f"{r['model']:32s} MAE {r['MAE']:>14,.0f}  R2 {r['R2']:10.4f}")

In [ ]:
# Three quantile models -> an 80% prediction interval.
# xgboost >= 2.0 has reg:quantileerror, so we use it (same hyperparameters as the
# other XGBoost models; much faster than sklearn's single-threaded GBR).
quantiles = [0.1, 0.5, 0.9]
q_preds = {}
t0 = time.time()
for q in quantiles:
    pipe = Pipeline([('prep', prep_tree),
                     ('model', XGBRegressor(objective='reg:quantileerror',
                                            quantile_alpha=q, n_estimators=300,
                                            max_depth=6, learning_rate=0.1,
                                            random_state=RANDOM_STATE, n_jobs=-1))])
    pipe.fit(X_train, y_train)
    q_preds[q] = pipe.predict(X_test)
    print(f'q={q} fitted')
print(f'three fits in {time.time() - t0:.0f}s')

# 1. Coverage check, overall and per cabin.
inside = (y_test >= q_preds[0.1]) & (y_test <= q_preds[0.9])
print(f'\ncoverage: {100 * inside.mean():.1f}%  (target ~80%)')
cls_test = X_test['class'].values
for cabin in ['Economy', 'Business']:
    mask = cls_test == cabin
    width = (q_preds[0.9] - q_preds[0.1])[mask]
    print(f'  {cabin:9s} coverage {100 * inside[mask].mean():5.1f}% | '
          f'median interval width {np.median(width):7,.0f} INR '
          f'({100 * np.median(width / q_preds[0.5][mask]):.0f}% of predicted median)')
print(f'below q0.1: {100 * (y_test < q_preds[0.1]).mean():.1f}% | '
      f'above q0.9: {100 * (y_test > q_preds[0.9]).mean():.1f}%  (each target ~10%)')
print(f'quantile crossings (q0.1 > q0.9): {(q_preds[0.1] > q_preds[0.9]).sum()}')

# 3. Pinball loss -- the metric that matches each loss.
print()
for q in quantiles:
    print(f'pinball loss q={q}: {mean_pinball_loss(y_test, q_preds[q], alpha=q):8,.1f}')
# For reference: the MSE model scored with the median's pinball loss.
mse_pred = loss_preds['MSE  (reg:squarederror)']
print(f'(MSE model, scored as q=0.5: {mean_pinball_loss(y_test, mse_pred, alpha=0.5):8,.1f})')

# 2. Interval for 40 test rows (20 Economy, 20 Business), sorted by predicted median.
rng = np.random.default_rng(RANDOM_STATE)
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, cabin in zip(axes, ['Economy', 'Business']):
    idx = rng.choice(np.flatnonzero(cls_test == cabin), 20, replace=False)
    idx = idx[np.argsort(q_preds[0.5][idx])]
    xs = np.arange(len(idx))
    ax.fill_between(xs, q_preds[0.1][idx], q_preds[0.9][idx], alpha=0.3,
                    label='80% interval (q0.1 - q0.9)')
    ax.plot(xs, q_preds[0.5][idx], color='C0', label='predicted median')
    ax.scatter(xs, y_test.values[idx], color='black', s=15, zorder=3, label='actual')
    ax.set_title(f'{cabin}: 20 test rows, sorted by predicted median')
    ax.set_xlabel('test row'); ax.set_ylabel('price (INR)'); ax.legend()
plt.tight_layout()
plt.show()

**Loss functions: your verdict.** *(5–7 sentences.)*

1. Which objective won on which metric? State the pattern you observed in one sentence.
2. Given the price distribution from Section 2a — two populations, strongly right-skewed overall — which loss do you think is the right default for *this* dataset, and why?
3. What was your 80% interval's actual coverage? If it missed the target, is the model over- or under-confident?
4. **The business question.** For a fare-recommendation tool, is it worse to suggest a price that is too high or too low? Pick a side, commit to it, and say which `α` you would ship. There is no neutral answer — the symmetric default *is* a choice, it is just an unexamined one.

1. **Pattern:** the squared-error model won **MAE (₹1,964), RMSE (₹3,487) and R² (0.9764)**. The absolute-error model won only **MAPE (13.1% vs 14.2%)**, and Huber (sklearn GBR) lost on every metric while taking **~100 s vs ~3 s**. (It also uses a different library and hyperparameters, so it is not a pure loss comparison.) So "each model wins its own metric" only half-held: the L1 model won the relative-error metric (it is better on cheap Economy fares) but lost even plain MAE, and the q = 0.5 quantile model's pinball loss (1,114) lost to the MSE model scored the same way (982). My reading: the within-segment residuals are not heavy-tailed (the big skew is *between* classes, and the features explain that), so MSE's outlier sensitivity costs little here. Meanwhile XGBoost's L1/quantile objectives have flat second derivatives, so with the same 300 trees at lr 0.1 they are effectively under-trained.
2. **Right default for this data:** squared error. The two-population skew lives in `class`, which the model sees directly, so MSE does not get dragged by outliers. If the stakeholder's metric is MAPE, the next thing I'd try is MSE on **log(price)** rather than switching to L1. I did not run that here.
3. **Coverage: 81.8%** (target 80%), with 9.1% below q0.1 and 9.1% above q0.9, and no quantile crossings. So it is very slightly conservative (a touch under-confident), which is essentially on target. By cabin: Economy 82.5% with an interval ~₹3.2k wide (58% of the predicted median), Business 80.4% at ~₹17k wide (32%). The interval is honest, but on Economy it is wide in relative terms.
4. **Business question:** for a fare-recommendation tool, quoting **too high is worse**. It loses the whole sale (and maybe the customer to a competitor on the same route), while quoting too low loses only part of the margin on one booking. I'd ship **α = 0.4**: lean deliberately low but stay close to the median, so we don't give away margin on flights that would have sold anyway. I'd revisit that per flight once load factor is available, because on a nearly full flight the asymmetry flips.

*Units lesson, measured:* XGBoost pseudo-Huber with the default `huber_slope=1` produced MAE ≈ ₹80 billion (R² ≈ −10¹³). Even `huber_slope=1000` still diverged (R² −168), and `huber_slope=10000` gave MAE ₹2,031 (R² 0.973), right in line with the other losses. The slope has to be on the scale of the errors you actually make.

---
# Section 6 — XGBoost vs Random Forest vs ARIMA

**Goal:** choose a model family on the basis of what the problem is, not what the leaderboard says.

These three are not three contestants in one race. **Two of them answer the same question in different ways, and the third answers a different question entirely** — recognising that is the actual skill in this section.

| Model | Family | What it needs | What it predicts |
|---|---|---|---|
| **Random Forest** | bagged trees | rows of features, any order | one target value **per row** |
| **XGBoost** | boosted trees | rows of features, any order | one target value **per row** |
| **ARIMA** | time series | one **ordered sequence**, equally spaced | the **next values** of that sequence |

Random Forest and XGBoost are interchangeable in their interface: both take `X, y` and predict a price for an individual ticket. ARIMA cannot do that at all — it does not take features. It takes a *sequence* and extends it.

---

### 6a — Bagging vs boosting: how RF and XGBoost actually differ

Both build many decision trees. **How** they build them is close to opposite.

| | **Random Forest (bagging)** | **XGBoost (boosting)** |
|---|---|---|
| Trees are built | **independently**, all in parallel | **sequentially**, each one after the last |
| Each tree is trained on | a bootstrap sample + a random feature subset | the **errors (gradients)** the ensemble still has |
| Individual trees are | deep, overfit, **decorrelated** | shallow, weak learners |
| Combined by | **averaging** | **adding** (scaled by the learning rate) |
| Mainly reduces | **variance** | **bias** |
| More trees → | never really worse, just slower | **can overfit**; needs `learning_rate` / early stopping |
| Tuning burden | low — works well out of the box | higher — more knobs that interact |
| Parallelises | trivially (trees are independent) | within a tree, not across them |

The intuition: **bagging** builds many different experts and averages away their individual mistakes. **Boosting** builds one expert, finds what it gets wrong, and hires a specialist for exactly that — repeatedly.

This explains their failure modes. A Random Forest rarely blows up but often plateaus slightly below a well-tuned booster. XGBoost usually wins *after tuning* and overfits cheerfully if you leave `n_estimators` high and `learning_rate` high.

In [ ]:
# Head-to-head, same pipeline, same split, with timings.
# RF is run UNBOUNDED as written (max_depth=None) on the full training set.
contenders = {
    'Dummy (mean)': DummyRegressor(strategy='mean'),
    'RandomForest': RandomForestRegressor(
        n_estimators=100, max_depth=None, min_samples_leaf=2,
        random_state=RANDOM_STATE, n_jobs=-1),
    'XGBoost': XGBRegressor(
        n_estimators=300, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        random_state=RANDOM_STATE, n_jobs=-1),
}
fitted = {}
for name, est in contenders.items():
    pipe = Pipeline([('prep', prep_tree), ('model', est)])
    t0 = time.time()
    pipe.fit(X_train, y_train)
    fit_s = time.time() - t0
    t0 = time.time()
    pred = pipe.predict(X_test)
    rep = regression_report(y_test, pred, name)
    rep['fit_seconds'] = round(fit_s, 1)
    rep['predict_seconds'] = round(time.time() - t0, 2)
    fitted[name] = pipe
    tbl = record(rep)
display(tbl.loc[list(contenders)])

rf = fitted['RandomForest'].named_steps['model']
print('RF mean tree depth:', np.mean([t.get_depth() for t in rf.estimators_]).round(1),
      '| mean leaves per tree:', int(np.mean([t.get_n_leaves() for t in rf.estimators_])))
n_nodes = sum(t.tree_.node_count for t in rf.estimators_)
print(f'RF total nodes: {n_nodes:,}  (~{n_nodes * 72 / 1e6:.0f} MB of tree arrays) '
      f'vs XGBoost: 300 trees of depth <= 6 (<= {300 * 127:,} nodes)')

**RF vs XGBoost.** *(4–6 sentences.)*

- Which won, and by how much in **MAE rupees** (not R² — rupees is the version a stakeholder understands)?
- How did fit times compare? If one took 10× longer for 1% more R², which would you ship, and what does that depend on?
- How far above the `DummyRegressor` is each? That gap is how much your features are actually worth.
- Which would you rather hand to a teammate who has never tuned a model, and why?

- **Random Forest won clearly:** MAE **₹1,096 vs ₹1,987** for XGBoost (₹890 less, −45%), R² 0.9859 vs 0.9761, MAPE **7.2% vs 14.2%**. This is not a 1% R² difference. In rupees it roughly halves the typical miss, and it does so on Economy too.
- **Cost:** RF fit in **~19 s vs ~2.5 s** (~8×) and predicted 60k rows in ~0.5 s vs ~0.15 s. It also grew 100 trees averaging depth ~42 and ~40,700 leaves each (8.1M nodes in total), against XGBoost's 300 depth-≤6 trees. So the "10× slower for 1%" trap does not apply here: 8× of a 20-second fit buys a 45% smaller error. The real cost is memory and prediction latency, and that only matters if this ran per-request at high volume.
- **Above the floor:** the Dummy's MAE is **₹19,769** (R² 0). RF removes 94% of that error and XGBoost 90%. That gap is what the features are worth.
- **Why the gap:** XGBoost at these settings is **underfit**: its CV train R² (0.9777) is barely above test (0.9765), the signature of bias rather than variance. Depth-6 trees can't carve out the many airline × route × days_left cells that RF's deep trees can. RF's deep trees may also benefit from the random split, which puts the same flight at neighbouring `days_left` values in both train and test. Its edge might shrink under a split by flight or by date, and I'd test that before trusting the full 45%.
- **For a teammate who has never tuned a model:** Random Forest. It got this result with near-default settings, more trees never make it worse, and there is no learning-rate/early-stopping interaction to get wrong. XGBoost needs a tuning pass (deeper trees, more rounds) to show what it can do. That is Deliverable 3.

### 6b — ARIMA: the right tool for a question we barely have

**ARIMA** = **A**uto**R**egressive **I**ntegrated **M**oving **A**verage, written `ARIMA(p, d, q)`:

- **AR(p)** — regress the value on its own previous `p` values. "Today looks like yesterday."
- **I(d)** — **difference** the series `d` times to make it **stationary** (constant mean/variance over time). A series that trends upward forever cannot be modelled directly; its *changes* usually can.
- **MA(q)** — regress on the previous `q` forecast *errors*. Lets recent shocks decay.

ARIMA is the standard tool for forecasting a single quantity through time: monthly revenue, daily passengers, weekly bookings.

### So does it apply to this dataset?

Be honest about this, because it is the real lesson:

**Mostly, no — and that is a legitimate finding.** ARIMA needs a series indexed by time, and this dataset **has no dates**. There is no departure date, no booking date, no timestamp anywhere. Without a calendar axis there is no seasonality, no trend over real time, no day-of-week effect. The dataset is a cross-section: 300,000 independent snapshots with no temporal order among them.

**But there is one real sequence in here**, and you plotted it in Section 2c: **mean price by `days_left`**. Days-to-departure is an ordered, equally-spaced index, so the booking curve is a genuine series of ~49 points — and forecasting the booking curve is a real revenue-management task. So we apply ARIMA to the thing it can actually model, and then we are precise about what we did and did not learn.

> **The transferable lesson:** your first model-selection question is never "which model scores best?" It is **"what shape is my data, and what shape does this model require?"** A model that needs a time index will not acquire one because you want it to. Picking the wrong family is a mistake no amount of hyperparameter tuning can repair.

In [ ]:
import warnings
from statsmodels.tsa.stattools import adfuller
warnings.filterwarnings('ignore', message='adfuller currently returns',
                        category=FutureWarning)

# Sort DESCENDING by days_left: position 0 is furthest from departure, the last
# position is the day before. Time now flows with the index.
curve = clean.groupby('days_left')['price'].mean().sort_index(ascending=False)
series = curve.reset_index(drop=True)
print(f'{len(series)} points  (days_left {curve.index.max()} -> {curve.index.min()})')

# Augmented Dickey-Fuller. H0 = "the series is NON-stationary".
stat, p, *_ = adfuller(series)
print(f'ADF p-value (raw):         {p:.4f}')
stat_d, p_d, *_ = adfuller(series.diff().dropna())
print(f'ADF p-value (differenced): {p_d:.4f}')
stat_d2, p_d2, *_ = adfuller(series.diff().diff().dropna())
print(f'ADF p-value (diff twice):  {p_d2:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
axes[0].plot(series, marker='o', ms=3)
axes[0].set_title('booking curve (position 0 = 49 days out)')
axes[0].set_xlabel('step toward departure'); axes[0].set_ylabel('mean price (INR)')
axes[1].plot(series.diff(), marker='o', ms=3, color='C1')
axes[1].axhline(0, color='grey', lw=0.8)
axes[1].set_title('first difference')
axes[1].set_xlabel('step toward departure'); axes[1].set_ylabel('change in mean price (INR)')
plt.tight_layout()
plt.show()

In [ ]:
import warnings
from statsmodels.tsa.arima.model import ARIMA

train_s, test_s = series[:-9], series[-9:]
print(f'{len(train_s)} training points, {len(test_s)} held out '
      f'(days_left {curve.index[-9]} -> {curve.index[-1]})')

# THE BASELINE THAT KEEPS YOU HONEST: carry the last observed value forward.
naive = np.repeat(train_s.iloc[-1], 9)
naive_mae = mean_absolute_error(test_s, naive)

# d=1 from the ADF tests above. Compare a few orders by AIC and by held-out MAE.
orders = [(2, 1, 2), (1, 1, 0), (0, 1, 1), (3, 1, 1), (1, 1, 1)]
arima_rows, forecasts = [], {}
for order in orders:
    t0 = time.time()
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        res = ARIMA(train_s, order=order).fit()
    fit_s = time.time() - t0
    forecasts[order] = np.asarray(res.forecast(steps=9))
    arima_rows.append({'order': str(order), 'AIC': res.aic,
                       'held-out MAE': mean_absolute_error(test_s, forecasts[order]),
                       'fit_seconds': round(fit_s, 2), 'warnings': len(caught)})
arima_table = pd.DataFrame(arima_rows).set_index('order')
arima_table.loc['naive last value'] = [np.nan, naive_mae, 0.0, 0]
display(arima_table.round(1))

# Full summary for the specification the notebook suggests.
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    arima = ARIMA(train_s, order=(2, 1, 2)).fit()
print(arima.summary())
fc = forecasts[(2, 1, 2)]
print('ARIMA(2,1,2) MAE:', round(mean_absolute_error(test_s, fc), 1))
print('Naive MAE:       ', round(naive_mae, 1))

order_of = {str(o): o for o in orders}
best_aic = arima_table['AIC'].idxmin()
plt.plot(train_s.index, train_s, marker='o', ms=3, label='training series')
plt.plot(test_s.index, test_s, marker='o', ms=4, color='black', label='actual (held out)')
plt.plot(test_s.index, fc, marker='x', label='ARIMA(2,1,2) forecast')
if best_aic != '(2, 1, 2)':
    plt.plot(test_s.index, forecasts[order_of[best_aic]], marker='x',
             label=f'ARIMA{best_aic} forecast (lowest AIC)')
plt.plot(test_s.index, naive, ls='--', label='naive last value')
plt.xlabel('step toward departure  (last point = 1 day out)')
plt.ylabel('mean price (INR)')
plt.title('booking-curve forecast of the final 9 days')
plt.legend()
plt.show()

**ARIMA: the verdict.** *(5–7 sentences. This is the most important written answer in the notebook.)*

1. Did ARIMA beat the naive last-value forecast? By how much? If it did not, say so plainly — a negative result reported honestly is worth more than a flattering one.
2. **What question did this ARIMA model answer?** Be precise. It did *not* price a flight. Write down what it actually produces and who could use it.
3. **What did you give up to use it?** You aggregated 300,000 rows into ~49 points. Name what was thrown away. Can this model tell you anything about airline, class, route or stops?
4. Would you include ARIMA in the deliverable to American Airlines? If so, in what role — alongside XGBoost rather than instead of it? If not, what would have to change about the dataset for it to earn a place?

1. **No meaningful win.** The naive last-value forecast scored MAE **₹1,650** on the final 9 days. The suggested ARIMA(2,1,2) did *worse* (**₹1,678**), threw convergence warnings and a near-singular covariance matrix, and its MA(2) term sits at −0.999, essentially cancelling its AR(2) term, which is a sign of an over-parameterised model. The best order, (3,1,1), scored ₹1,631, only 1% better than naive and well within noise for 9 points. The lowest-AIC order, (1,1,1), lost to naive (₹1,663). I used d = 1, but I can't claim stationarity: the ADF test never rejected (p = 0.91 raw, 0.29 after one difference, 0.21 after two). With 49 points ADF has little power, and the series has a regime change about 15 days out. Part of the held-out tail is also unforecastable from this series alone: the day-1 drop from ₹30.2k to ₹21.6k is the Business share falling from 35% to 14%, a cabin-mix change no univariate model can see coming.
2. **What it answered:** "given the average fare across *all* airlines, cabins and routes in this scrape at 49→10 days out, what is that average at 9→1 days out?" It produces **one number per day-to-departure for the whole market**, a booking-curve forecast. A revenue-management analyst might use that shape, but it does not price a single ticket.
3. **What I gave up:** I collapsed **300,153 rows into 49 means**, discarding airline, class, route, stops, departure/arrival time and duration entirely. The model can say nothing about any of them, and the class mix even contaminates the one thing it does model.
4. **Would I include it?** Not as a pricing model. At most it belongs in an appendix as a "booking-curve shape" view alongside the per-row tree model, not instead of it. And honestly the per-class curves from 2c tell that story better than the ARIMA does. For ARIMA to earn a place, the data would need **real calendar time**: e.g. fares or bookings per route per departure date over months, so there is a long, equally spaced series with trend and seasonality for SARIMAX to model.

> **For the curious: SARIMAX.** The `statsmodels` family does extend to seasonality and features: `SARIMAX` adds a seasonal component and an `exog=` argument for external regressors. With real departure dates you could model weekly and holiday seasonality and feed in fuel price or capacity — which is much closer to how an airline forecasts demand. We cannot, because this dataset has no dates. Worth knowing the door exists.

### 6c — The model-selection decision table

Fill this in from your own numbers. This table, not any single score, is the deliverable for Section 6.

| Model | Best MAE (INR) | R² | Fit time | Interpretable? | When would you choose it? |
|---|---|---|---|---|---|
| Dummy (mean) | 19,769 | 0.000 | 0.3 s | — | never; it is the floor |
| Ridge | 4,499 (any alpha) | 0.910 | ~0.4 s | yes — coefficients | when a stakeholder needs to see *why* a price is what it is and every feature should keep a say; never as the price itself (MAPE ~42%) |
| Lasso | 4,406 (alpha = 100) | 0.908 | ~1–2 s | yes — sparse coefficients | the explainer model: 16 readable coefficients, and it tells you which dummies don't matter (3 budget airlines, 3 cities) |
| Random Forest | 1,096 (CV ₹1,086 ± 6) | 0.986 | ~19 s | no | the per-ticket price model today: best accuracy with no tuning; watch memory (~590 MB) and prediction latency |
| XGBoost | 1,964 (MSE objective; CV 1,975 ± 20 for the subsampled config) | 0.976 | ~2.5–3 s | no | when fit/predict cost or model size matters, or after a tuning pass (it is underfit at depth 6) |
| ARIMA | n/a — different target (booking-curve MAE 1,631–1,678 vs naive 1,650) | n/a | ≤ 0.1 s per order | partly | only for forecasting an aggregate curve through real calendar time, which this data doesn't have |

**The one you would ship, and the three reasons why:** **Random Forest**, as the per-ticket fare model, with the Lasso alongside it as the explanation layer.

1. **Error in rupees, where it matters:** MAPE 7.2% vs 14.2% for XGBoost and ~42% for the linear models. An Economy fare quoted within ~7% instead of ~14% is a different product, not a decimal place.
2. **Low tuning burden:** this came from near-default settings. XGBoost at the same effort is visibly underfit (train R² ≈ test R²) and needs a search to compete. Shipping the model that already works, and has fewer knobs to break, is cheaper.
3. **Robust and cheap enough:** bagging averages 100 decorrelated trees, so adding trees can't make it overfit, and CV variation is small (₹1,086 ± 6). A 20-second fit is trivial for a nightly retrain.

The strongest argument against: it is heavy (~8.1M nodes) and slower to predict. It cannot extrapolate beyond prices it has seen. And part of its lead may come from memorising near-identical flights at adjacent `days_left` under a random split. A tuned XGBoost could match it at a fraction of the size.

---
# Section 7 — Defend your choice

### 7a — The full scoreboard

In [ ]:
# TODO: Print the complete scoreboard and sort it by MAE.
#
#   show_scoreboard().sort_values('MAE')
#
# Then write the one-line summary a PM would actually read:
#   "Best model: ____, MAE ₹____ (±__ across CV folds), vs ₹____ for the dummy."

### 7b — What this means for American Airlines

We are modelling **Indian domestic fares scraped from EaseMyTrip in 2022**. American Airlines is a different carrier in a different market in a different year. Be precise about what transfers.

**What transfers — the structure of the problem and the method.** Airline revenue management has the same shape everywhere: fares move with the booking horizon, cabin class dominates price, route density and competition set the base level, and time-of-day and connections differentiate the product. Your model-selection reasoning transfers completely: tree ensembles for per-ticket prediction, asymmetric loss where over- and under-pricing cost differently, a time-series model for forecasting an aggregate curve through real time.

**What does not transfer:** nearly all the specifics. Different regulatory regime, competitive set, cost base, currency, and demand seasonality — and 2022 was still distorted by post-pandemic recovery.

**What the dataset is missing that real pricing requires:**

| Missing | Why it matters |
|---|---|
| **Seat inventory / load factor** | Revenue management is fundamentally about *remaining seats*. We have no capacity data at all — the largest single gap. |
| **Departure dates** | We have `days_left` but not *when*. No seasonality, no holidays, no day-of-week. **This is also precisely why ARIMA had so little to work with.** |
| **Competitor fares** | Fares are set against what everyone else charges on the route, in near-real-time. |
| **Fare class / fare basis** | "Economy" spans a dozen booking classes with different prices and restrictions; our `class` flattens all of it to two values. |
| **Demand signals** | Searches, booking pace, events — the leading indicators pricing teams actually watch. |

### 7c — Reflection

Answer each in the cell below it, in your own words. A PM will ask these in your PR review, so treat it as practice for that conversation.

---

**Q1 — The honest number.** You produced several R² values for the same task. List them with one line each on what changed, then name the single number you would put in front of American Airlines and defend it in two sentences.

**Answer Q1:** *(Write here)*

**Q2 — Model family.** You would ship one of Ridge / Lasso / Random Forest / XGBoost. Name it, and give **three** reasons that are not "it had the best score" — e.g. fit time, tuning burden, interpretability, robustness, what the stakeholder needs. Then name the strongest argument *against* your choice.

**Answer Q2:** *(Write here)*

**Q3 — Lasso vs Ridge.** Explain to a teammate who has not done this deliverable why L1 produces exact zeros and L2 does not. Then say which you would use on this dataset and why — including the possibility that neither penalty mattered much here, and what about this dataset makes that the case.

**Answer Q3:** *(Write here)*

**Q4 — Loss functions.** Your Deliverable 1 model minimised squared error because that is the default. Having seen the comparison: was that the right default for this data? Which loss would you train on now, and which metric would you report? If those two differ, justify the mismatch.

**Answer Q4:** *(Write here)*

**Q5 — ARIMA and problem shape.** Someone proposes ARIMA for predicting individual ticket prices. Explain in three sentences why that does not work, and what they would need to change about the data or the question for a time-series model to be the right call.

**Answer Q5:** *(Write here)*

**Q6 — Leakage.** Explain leakage to a teammate using an example from your own notebook run (not one from the text). Then name the one habit from this notebook you will carry into Deliverable 3 to prevent it.

**Answer Q6:** *(Write here)*

**Q7 — Team git.** Deliverable 1 was `deliverable1/<your-name>` → `main`. Deliverable 2 is `team-##/<your-name>` → `team-##/main` → `main`. What does the extra layer buy a project with 70+ contributors? Then describe something that actually went wrong or felt awkward in your team's git this week — a real friction point is worth more here than a clean answer.

**Answer Q7:** *(Write here)*

---
## Before you submit — checklist

**The notebook**

1. **Kernel → Restart & Run All.** It must run top-to-bottom with no errors. A notebook that only works in the order you happened to click cells is not reproducible.
2. Every `# TODO` cell is filled in.
3. Every markdown answer prompt has a real answer. "*(Write here)*" left anywhere is an incomplete deliverable.
4. The three summary tables are filled in: the EDA decisions (2e), the model-selection table (6c), and the scoreboard (7a).
5. **Kernel → Restart & Clear Output** before committing. Keeps the diff reviewable and prevents most notebook merge conflicts.

**Git hygiene**

6. `git status` shows **no** `.csv`. If it does, stop — do not commit, and check [`.gitignore`](../.gitignore).
7. You are on your own member branch — not your team's trunk, and definitely not `main`. Check with `git branch --show-current`.

```sh
git add deliverable2/flight_price_modeling.ipynb
git commit -m "deliverable2: EDA, leakage fixes, model selection"
git push
```

8. Open a PR from your member branch into **your team's trunk branch**. GitHub defaults the base to `main` — change it by hand.
9. Request review from the teammate assigned to you in [`README.md`](README.md), and **review the PR assigned to you**. Reviewing is half the job.
10. Once every member's PR has merged into the team trunk, the team fills in [`TEAM_REPORT.md`](TEAM_REPORT.md) together and a PM reviews the trunk → `main`.

---

You arrived with one model and one score. You are leaving with five models, four loss functions, an honest number, and — most importantly — a reason for every choice.

In Deliverable 3 we tune the winner: hyperparameter search over the pipeline you built, learning curves to distinguish a data problem from a model problem, and error analysis by segment to find out *where* it fails rather than just how much. 🛫